# IEEE118 EMT timing: conventional components vs SSN

Fresh runs use Δt = 50 µs and T_sim = 30 s with detailed scheduler profiling. SSN, aggregate, the earlier grouped baseline, and grouped shared maps each have five sequential runs. Per user direction, Conventional has one completed run; an in-progress second run was stopped and is excluded. Detailed solve/pre/post and per-step statistics come from the median-wall-time run for each configuration.

In [ ]:
import lzma
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DT = 50e-6
FINAL_TIME = 30.0
EXPECTED_STEPS = 599_999
DATA = Path("examples/Notebooks/Benchmarks/data/verified_rerun")
if not DATA.exists():
    DATA = Path("data/verified_rerun")

CONFIG_LABELS = {
    "ssn": "SSN",
    "conventional": "Conventional",
    "aggregate": "Aggregate",
    "grouped_passive_huge_ssn": "Grouped passive huge SSN — previous",
    "grouped_shared_maps": "Grouped passive huge SSN — shared maps",
}
COLORS = {
    "SSN": "#F58518",
    "Conventional": "#4C78A8",
    "Aggregate": "#59A14F",
    "Grouped passive huge SSN — previous": "#B279A2",
    "Grouped passive huge SSN — shared maps": "#E15759",
}

In [ ]:
# Wall-clock times from the actual sequential reruns; conventional is one run.
wall = pd.read_csv(DATA / "wall_times.csv")
wall["configuration_name"] = wall["configuration"].map(CONFIG_LABELS)
run_counts = wall.groupby("configuration_name")["wall_s"].agg(
    ["count", "mean", "std", "min", "max"]
)
run_counts["relative_to_conventional_%"] = (
    run_counts["mean"] / run_counts.loc["Conventional", "mean"] - 1
) * 100
run_counts.round(3)

In [ ]:
# Detailed stage and per-step data from the selected representative run for each configuration.
def load_profile(configuration, label):
    tasks = pd.read_csv(
        DATA / f"{configuration}_TaskTimes.csv",
        header=None,
        names=["task", "avg_ns", "min_ns", "max_ns", "calls"],
    )
    tasks["stage"] = tasks["task"].str.rsplit(".", n=1).str[-1]
    tasks["cumulative_s"] = tasks["avg_ns"] * tasks["calls"] / 1e9
    stage = tasks.groupby("stage")["cumulative_s"].sum()
    step_path = DATA / f"{configuration}_StepTimes.log.xz"
    if step_path.exists():
        with lzma.open(step_path, "rt") as step_file:
            step_ms = np.loadtxt(step_file, skiprows=1) * 1000.0
    else:
        step_ms = (
            np.loadtxt(DATA / f"{configuration}_StepTimes.log", skiprows=1) * 1000.0
        )
    return {
        "configuration": label,
        "steps": step_ms.size,
        "step_mean_ms": step_ms.mean(),
        "step_median_ms": np.median(step_ms),
        "step_p95_ms": np.percentile(step_ms, 95),
        "step_max_ms": step_ms.max(),
        "solver_cumulative_s": stage.get("Solve", 0.0),
        "pre_step_cumulative_s": stage.get("MnaPreStep", 0.0),
        "post_step_cumulative_s": stage.get("MnaPostStep", 0.0),
    }


profile = pd.DataFrame(
    [load_profile(key, label) for key, label in CONFIG_LABELS.items()]
)
profile["tracked_task_total_s"] = profile[
    ["solver_cumulative_s", "pre_step_cumulative_s", "post_step_cumulative_s"]
].sum(axis=1)
profile.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

for label, group in wall.groupby("configuration_name", sort=False):
    axes[0].plot(group["run"], group["wall_s"], "o-", label=label, color=COLORS[label])
axes[0].set(
    title="Wall-clock time (5 runs where available)", xlabel="Run", ylabel="Seconds"
)
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)
axes[0].set_xticks([1, 2, 3, 4, 5])

x = np.arange(len(profile))
width = 0.22
for j, col in enumerate(["step_mean_ms", "step_median_ms", "step_p95_ms"]):
    axes[1].bar(
        x + (j - 1) * width,
        profile[col],
        width,
        label=col.replace("step_", "").replace("_", " "),
    )
axes[1].set(
    title="Simulation-step timing",
    ylabel="Milliseconds",
    xticks=x,
    xticklabels=profile["configuration"],
)
axes[1].tick_params(axis="x", rotation=18)
axes[1].grid(axis="y", alpha=0.25)
axes[1].legend(fontsize=8)

profile.set_index("configuration")[
    ["solver_cumulative_s", "pre_step_cumulative_s", "post_step_cumulative_s"]
].plot(kind="bar", ax=axes[2], color=["#E45756", "#72B7B2", "#54A24B"])
axes[2].set(title="Cumulative profiled task time", xlabel="", ylabel="Seconds")
axes[2].tick_params(axis="x", rotation=18)
axes[2].grid(axis="y", alpha=0.25)
axes[2].legend(["Solve", "Pre-step", "Post-step"], fontsize=8)
fig.suptitle("IEEE118 EMT | Δt = 50 µs | T_sim = 30 s", fontsize=14)
plt.show()

In [ ]:
# Reproducibility and sample-count checks.
assert DT == 50e-6
assert FINAL_TIME == 30.0
assert (profile["steps"] == EXPECTED_STEPS).all()
assert wall.groupby("configuration").size().to_dict() == {
    "aggregate": 5,
    "conventional": 1,
    "grouped_passive_huge_ssn": 5,
    "grouped_shared_maps": 5,
    "ssn": 5,
}
assert set(profile["configuration"]) == set(CONFIG_LABELS.values())
print("Wall-time means (s) and run counts:")
display(run_counts[["count", "mean", "std"]].round(3))
print(
    "Conventional is intentionally a single completed run; no mean over interrupted runs is used."
)
print(
    "All detailed profiles contain 599,999 measured simulation steps. The grouped shared-map profile is the median of five matching 30-second runs."
)